# DNAscape mapping atlas

This notebook is organised by **target feature**: first decide what you want to infer, then provide the data needed to get it. In `rmap`, `source` contains the supplied replication data or kinetic quantities, while `assumptions` contains settings such as resolution, `n_bins`, geometry, target form, and fitting controls.

We use timing data as the entry point and move through three targets: origin firing, fork speed, and replication timing. The arrays used by the model are kinetic interpretations rather than raw assay values, so they should be preprocessed with the biological question in mind. See `1_datamanagement.ipynb` for data-management details.

## Setup

Autoreload keeps notebook runs aligned with source-code edits.

In [ ]:
%reload_ext autoreload
%autoreload 2

from dnascape import *

## Example data

We use the bundled hESC chromosome 1 replication-timing profile. The selected profile is a one-dimensional `T(x)` array, and `positions_kb` gives the matching genomic coordinates.

In [ ]:
chr_number = 1
output_resolution_kb = 1
interval_kb = (0, 60_000)
fork_speed = 1.4
simulation_replicates = 100

replication_timing_full = loadcsv(
    "example",
    example="replication_timing_hESC",
    chr_number=chr_number,
)

replication_timing, positions_kb = select_profile_region(
    replication_timing_full,
    output_resolution_kb=output_resolution_kb,
    interval_kb=interval_kb,
)

n_bins = replication_timing.size

In [ ]:
plotf(
    replication_timing,
    x_array=positions_kb,
    invyQ=True,
    xtitle="Chromosome position (kb)",
    ytitle="Replication timing (min)",
)

## Core mapping routes

The table summarises the central kinetic triangle: timing `T(x)`, firing `I`, and fork speed `v`. Constant, spatial, time-only, and space-time forms are labelled as `I0`, `I(x)`, `I(t)`, `I(x,t)` and `v0`, `v(x)`, `v(t)`, `v(x,t)`.

Rows marked **implemented** are available now. The current deterministic methods cover the constant-speed cases involving `I0` or `I(x)`. In the examples below, every route is plotted with the same row order: origin firing, fork speed, and replication timing. Labels mark the target, and the timing row includes an `rsim` timing trace generated from the same firing-rate and fork-speed inputs.

In [ ]:
I_FORMS = ["I0", "I(x)", "I(t)", "I(x,t)"]
V_FORMS = ["v0", "v(x)", "v(t)", "v(x,t)"]

rows = []

for target_form in I_FORMS:
    for fork_speed_form in V_FORMS:
        implemented = target_form in {"I0", "I(x)"} and fork_speed_form == "v0"
        if target_form == "I0" and implemented:
            method = "timing_and_speed_to_constant_firing_rate"
            assumptions = "resolution, target_form='I0'"
        elif target_form == "I(x)" and implemented:
            method = "timing_and_speed_to_firing_rate"
            assumptions = "resolution"
        else:
            method = "-"
            assumptions = "resolution, target_form"
        rows.append({
            "target": "firing_rate",
            "target form": target_form,
            "required data": "replication_timing, fork_speed",
            "firing data": "-",
            "fork-speed data": fork_speed_form,
            "assumptions": assumptions,
            "method": method,
            "status": "implemented" if implemented else "needs method",
        })

for target_form in V_FORMS:
    for firing_form in I_FORMS:
        implemented = target_form in {"v0", "v(x)"} and firing_form in {"I0", "I(x)"}
        if target_form == "v0" and implemented:
            method = "timing_and_firing_to_constant_fork_speed"
            assumptions = "resolution, target_form='v0'"
        elif target_form == "v(x)" and implemented:
            method = "timing_and_firing_to_spatial_fork_speed"
            assumptions = "resolution, target_form='v(x)', smoothw"
        else:
            method = "-"
            assumptions = "resolution, target_form"
        rows.append({
            "target": "fork_speed",
            "target form": target_form,
            "required data": "replication_timing, firing_rate",
            "firing data": firing_form,
            "fork-speed data": "-",
            "assumptions": assumptions,
            "method": method,
            "status": "implemented" if implemented else "needs method",
        })

for firing_form in I_FORMS:
    for fork_speed_form in V_FORMS:
        implemented = firing_form in {"I0", "I(x)"} and fork_speed_form in {"v0", "v(x)"}
        assumptions = "resolution, n_bins" if firing_form == "I0" else "resolution"
        rows.append({
            "target": "replication_timing",
            "target form": "T(x)",
            "required data": "firing_rate, fork_speed",
            "firing data": firing_form,
            "fork-speed data": fork_speed_form,
            "assumptions": assumptions,
            "method": "firing_and_speed_to_timing_deterministic" if implemented else "-",
            "status": "implemented" if implemented else "needs method",
        })

mapping_routes = pd.DataFrame(rows)
mapping_routes

## 1. Target: origin firing

Use this target when timing is measured and fork speed is fixed or supplied. With `v0`, DNAscape can infer either a scalar baseline `I0` or a spatial profile `I(x)`.

### 1.1 Timing and v0 to I0

This gives a single initiation-rate baseline for the whole interval.

In [ ]:
constant_firing_fit = rmap(
    source={
        "replication_timing": replication_timing,
        "fork_speed": fork_speed,
    },
    target="firing_rate",
    assumptions={
        "resolution": output_resolution_kb,
        "target_form": "I0",
        "maxiter": 6,
    },
    return_info=True,
)

In [ ]:
fitted_I0 = constant_firing_fit["values"]
sim_timing_I0_v0 = rsim(
    source={
        "firing_rate": fitted_I0,
        "fork_speed": fork_speed,
    },
    assumptions={
        "sim_number": simulation_replicates,
        "resolution": output_resolution_kb,
        "n_bins": n_bins,
        "seed": 11,
        "verbose": False,
    },
)["replication_timing"]

print(constant_firing_fit["method"], fitted_I0)

In [ ]:
mplotf(
    [
        np.full(n_bins, fitted_I0),
        np.full(n_bins, fork_speed),
        [replication_timing, sim_timing_I0_v0],
    ],
    mlabels=[
        ["Target: fitted I0"],
        ["Input: assumed v0"],
        ["Input: timing data", "rsim: fitted I0 + v0"],
    ],
    mx_array=[positions_kb, positions_kb, positions_kb],
    minvyQ=[False, False, True],
    mxtitle=["", "", "Chromosome position (kb)"],
    mytitle=["Origin firing", "Fork speed", "Replication timing (min)"],
    figsize=(10, 1.9),
    hspace=0.16,
)

### 1.2 Timing and v0 to I(x)

This is the richer inverse map, returning one firing-rate value per genomic bin.

In [ ]:
spatial_firing_fit = rmap(
    source={
        "replication_timing": replication_timing,
        "fork_speed": fork_speed,
    },
    target="firing_rate",
    assumptions={
        "resolution": output_resolution_kb,
        "maxiter": 6,
    },
    return_info=True,
)

In [ ]:
firing_rate = spatial_firing_fit["values"]
sim_timing_Ix_v0 = rsim(
    source={
        "firing_rate": firing_rate,
        "fork_speed": fork_speed,
    },
    assumptions={
        "sim_number": simulation_replicates,
        "resolution": output_resolution_kb,
        "n_bins": n_bins,
        "seed": 12,
        "verbose": False,
    },
)["replication_timing"]

print(spatial_firing_fit["method"], firing_rate.shape)

In [ ]:
mplotf(
    [
        firing_rate,
        np.full(n_bins, fork_speed),
        [replication_timing, sim_timing_Ix_v0],
    ],
    mlabels=[
        ["Target: fitted I(x)"],
        ["Input: assumed v0"],
        ["Input: timing data", "rsim: fitted I(x) + v0"],
    ],
    mx_array=[positions_kb, positions_kb, positions_kb],
    minvyQ=[False, False, True],
    mxtitle=["", "", "Chromosome position (kb)"],
    mytitle=["Origin firing", "Fork speed", "Replication timing (min)"],
    figsize=(10, 1.9),
    hspace=0.16,
)

## 2. Target: fork speed

Use this target when timing and firing are supplied. DNAscape can fit either one constant speed `v0` or a spatial effective speed profile `v(x)` when the supplied firing field is `I0` or `I(x)`.

### 2.1 Timing and I0 to v0

This estimates the constant fork speed implied by timing and the scalar firing baseline.

In [ ]:
constant_speed_from_I0 = rmap(
    source={
        "replication_timing": replication_timing,
        "firing_rate": fitted_I0,
    },
    target="fork_speed",
    assumptions={
        "resolution": output_resolution_kb,
        "target_form": "v0",
        "maxiter": 6,
        "initial_fork_speed": fork_speed,
    },
    return_info=True,
)

In [ ]:
fitted_v0_from_I0 = constant_speed_from_I0["values"]
sim_timing_I0_fitv0 = rsim(
    source={
        "firing_rate": fitted_I0,
        "fork_speed": fitted_v0_from_I0,
    },
    assumptions={
        "sim_number": simulation_replicates,
        "resolution": output_resolution_kb,
        "n_bins": n_bins,
        "seed": 21,
        "verbose": False,
    },
)["replication_timing"]

print(constant_speed_from_I0["method"], fitted_v0_from_I0)

In [ ]:
mplotf(
    [
        np.full(n_bins, fitted_I0),
        np.full(n_bins, fitted_v0_from_I0),
        [replication_timing, sim_timing_I0_fitv0],
    ],
    mlabels=[
        ["Input: fitted I0"],
        ["Target: fitted v0"],
        ["Input: timing data", "rsim: I0 + fitted v0"],
    ],
    mx_array=[positions_kb, positions_kb, positions_kb],
    minvyQ=[False, False, True],
    mxtitle=["", "", "Chromosome position (kb)"],
    mytitle=["Origin firing", "Fork speed", "Replication timing (min)"],
    figsize=(10, 1.9),
    hspace=0.16,
)

### 2.2 Timing and I(x) to v0

Here the same scalar fork-speed fit is performed using the spatial firing profile.

In [ ]:
constant_speed_from_Ix = rmap(
    source={
        "replication_timing": replication_timing,
        "firing_rate": firing_rate,
    },
    target="fork_speed",
    assumptions={
        "resolution": output_resolution_kb,
        "target_form": "v0",
        "maxiter": 10,
        "initial_fork_speed": fork_speed,
    },
    return_info=True,
)

In [ ]:
fitted_v0_from_Ix = constant_speed_from_Ix["values"]
sim_timing_Ix_fitv0 = rsim(
    source={
        "firing_rate": firing_rate,
        "fork_speed": fitted_v0_from_Ix,
    },
    assumptions={
        "sim_number": simulation_replicates,
        "resolution": output_resolution_kb,
        "n_bins": n_bins,
        "seed": 22,
        "verbose": False,
    },
)["replication_timing"]

print(constant_speed_from_Ix["method"], fitted_v0_from_Ix)

In [ ]:
mplotf(
    [
        firing_rate,
        np.full(n_bins, fitted_v0_from_Ix),
        [replication_timing, sim_timing_Ix_fitv0, sim_timing_Ix_v0],
    ],
    mlabels=[
        None,
        None,
        ["Input: timing data", f"Timing (v = {fitted_v0_from_Ix:.2f} kb/min)", f"Timing (v = {fork_speed:.2f} kb/min)"],
    ],
    mx_array=[positions_kb, positions_kb, positions_kb],
    minvyQ=[False, False, True],
    mxtitle=["", "", "Chromosome position (kb)"],
    mytitle=["Origin firing", "Fork speed", "Replication timing (min)"],
    figsize=(10, 1.9),
    hspace=0.16,
)

### 2.3 Timing and I0 to v(x)

This fit asks whether regional fork-speed variation can explain timing when initiation is represented by a single baseline.

In [ ]:
spatial_speed_from_I0 = rmap(
    source={
        "replication_timing": replication_timing,
        "firing_rate": fitted_I0,
    },
    target="fork_speed",
    assumptions={
        "resolution": output_resolution_kb,
        "target_form": "v(x)",
        "maxiter": 40,
        "initial_fork_speed": fitted_v0_from_I0,
        "smoothw": 100,
    },
    return_info=True,
)

In [ ]:
fitted_vx_from_I0 = spatial_speed_from_I0["values"]
sim_timing_I0_fitvx = rsim(
    source={
        "firing_rate": fitted_I0,
        "fork_speed": fitted_vx_from_I0,
    },
    assumptions={
        "sim_number": simulation_replicates,
        "resolution": output_resolution_kb,
        "n_bins": n_bins,
        "seed": 23,
        "verbose": False,
    },
)["replication_timing"]

print(spatial_speed_from_I0["method"], fitted_vx_from_I0.shape)

In [ ]:
mplotf(
    [
        np.full(n_bins, fitted_I0),
        fitted_vx_from_I0,
        [replication_timing, sim_timing_I0_fitvx],
    ],
    mlabels=[
        ["Input: fitted I0"],
        ["Target: fitted v(x)"],
        ["Input: timing data", "rsim: I0 + fitted v(x)"],
    ],
    mx_array=[positions_kb, positions_kb, positions_kb],
    minvyQ=[False, False, True],
    mxtitle=["", "", "Chromosome position (kb)"],
    mytitle=["Origin firing", "Fork speed", "Replication timing (min)"],
    figsize=(10, 1.9),
    hspace=0.16,
)

### 2.4 Timing and I(x) to v(x)

Here both initiation and fork speed are spatial profiles, giving the most flexible implemented fork-speed map in this notebook.

In [ ]:
spatial_speed_from_Ix = rmap(
    source={
        "replication_timing": replication_timing,
        "firing_rate": firing_rate,
    },
    target="fork_speed",
    assumptions={
        "resolution": output_resolution_kb,
        "target_form": "v(x)",
        "maxiter": 100,
        "initial_fork_speed": fitted_v0_from_Ix,
        "smoothw": 100,
    },
    return_info=True,
)

In [ ]:
fitted_vx_from_Ix = spatial_speed_from_Ix["values"]
sim_timing_Ix_fitvx = rsim(
    source={
        "firing_rate": firing_rate,
        "fork_speed": fitted_vx_from_Ix,
    },
    assumptions={
        "sim_number": simulation_replicates,
        "resolution": output_resolution_kb,
        "n_bins": n_bins,
        "seed": 24,
        "verbose": False,
    },
)["replication_timing"]

print(spatial_speed_from_Ix["method"], fitted_vx_from_Ix.shape)

In [ ]:
mplotf(
    [
        firing_rate,
        fitted_vx_from_Ix,
        [replication_timing, sim_timing_Ix_fitvx],
    ],
    mlabels=[
        ["Input: fitted I(x)"],
        ["Target: fitted v(x)"],
        ["Input: timing data", "rsim: I(x) + fitted v(x)"],
    ],
    mx_array=[positions_kb, positions_kb, positions_kb],
    minvyQ=[False, False, True],
    mxtitle=["", "", "Chromosome position (kb)"],
    mytitle=["Origin firing", "Fork speed", "Replication timing (min)"],
    figsize=(10, 1.9),
    hspace=0.16,
)

## 3. Target: replication timing

Use this target when firing and fork speed are supplied and the goal is a predicted timing profile.

### 3.1 I0 and v0 to timing

A scalar firing rate needs `n_bins` so DNAscape knows the length of the returned profile.

In [ ]:
constant_timing_fit = rmap(
    source={
        "firing_rate": fitted_I0,
        "fork_speed": fitted_v0_from_I0,
    },
    target="replication_timing",
    assumptions={
        "resolution": output_resolution_kb,
        "n_bins": n_bins,
    },
    return_info=True,
)

timing_from_I0 = constant_timing_fit["values"]
sim_timing_from_I0 = rsim(
    source={
        "firing_rate": fitted_I0,
        "fork_speed": fitted_v0_from_I0,
    },
    assumptions={
        "sim_number": simulation_replicates,
        "resolution": output_resolution_kb,
        "n_bins": n_bins,
        "seed": 31,
        "verbose": False,
    },
)["replication_timing"]

print(constant_timing_fit["method"], timing_from_I0.shape)

In [ ]:
mplotf(
    [
        np.full(n_bins, fitted_I0),
        np.full(n_bins, fitted_v0_from_I0),
        [replication_timing, timing_from_I0, sim_timing_from_I0],
    ],
    mlabels=[
        ["Input: fitted I0"],
        ["Input: fitted v0"],
        ["Reference: timing data", "Target: timing from I0", "rsim: I0 + v0"],
    ],
    mx_array=[positions_kb, positions_kb, positions_kb],
    minvyQ=[False, False, True],
    mxtitle=["", "", "Chromosome position (kb)"],
    mytitle=["Origin firing", "Fork speed", "Replication timing (min)"],
    figsize=(10, 1.9),
    hspace=0.16,
)

### 3.2 I(x) and v0 to timing

The spatial firing profile maps forward to a timing profile over the same genomic bins.

In [ ]:
spatial_timing_fit = rmap(
    source={
        "firing_rate": firing_rate,
        "fork_speed": fitted_v0_from_Ix,
    },
    target="replication_timing",
    assumptions={
        "resolution": output_resolution_kb,
    },
    return_info=True,
)

timing_from_Ix = spatial_timing_fit["values"]
sim_timing_from_Ix = rsim(
    source={
        "firing_rate": firing_rate,
        "fork_speed": fitted_v0_from_Ix,
    },
    assumptions={
        "sim_number": simulation_replicates,
        "resolution": output_resolution_kb,
        "n_bins": n_bins,
        "seed": 32,
        "verbose": False,
    },
)["replication_timing"]

print(spatial_timing_fit["method"], timing_from_Ix.shape)

In [ ]:
mplotf(
    [
        firing_rate,
        np.full(n_bins, fitted_v0_from_Ix),
        [replication_timing, timing_from_Ix, sim_timing_from_Ix],
    ],
    mlabels=[
        ["Input: fitted I(x)"],
        ["Input: fitted v0"],
        ["Reference: timing data", "Target: timing from I(x)", "rsim: I(x) + v0"],
    ],
    mx_array=[positions_kb, positions_kb, positions_kb],
    minvyQ=[False, False, True],
    mxtitle=["", "", "Chromosome position (kb)"],
    mytitle=["Origin firing", "Fork speed", "Replication timing (min)"],
    figsize=(10, 1.9),
    hspace=0.16,
)

## Combined view

This final panel compares the constant and spatial routes on the same interval.

In [ ]:
mplotf(
    [
        [np.full(n_bins, fitted_I0), firing_rate],
        [np.full(n_bins, fitted_v0_from_I0), np.full(n_bins, fitted_v0_from_Ix)],
        [replication_timing, timing_from_I0, timing_from_Ix, sim_timing_from_I0, sim_timing_from_Ix],
    ],
    mlabels=[
        ["Fitted I0", "Fitted I(x)"],
        ["v0 from I0", "v0 from I(x)"],
        ["Timing data", "Timing from I0", "Timing from I(x)", "rsim: I0 + v0", "rsim: I(x) + v0"],
    ],
    mx_array=[positions_kb, positions_kb, positions_kb],
    minvyQ=[False, False, True],
    mxtitle=["", "", "Chromosome position (kb)"],
    mytitle=[
        "Firing rate",
        "Fork speed",
        "Replication timing (min)",
    ],
    figsize=(10, 2.3),
    hspace=0.18,
)